In [ ]:
import pandas as pd

In [ ]:
from pathlib import Path


def find_local_csv(filename: str) -> Path:
    """Find a CSV in common local notebook/project locations."""
    cwd = Path.cwd()

    # Most likely locations first.
    candidates = [
        cwd / filename,
        cwd / "MuseumsNotebook" / filename,
        cwd.parent / "MuseumsNotebook" / filename,
        cwd.parent / filename,
    ]

    # Also check a small recursive scope near cwd.
    search_roots = [cwd, cwd.parent]
    for root in search_roots:
        try:
            for match in root.rglob(filename):
                candidates.append(match)
        except Exception:
            pass

    seen = set()
    for file_path in candidates:
        resolved = file_path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if resolved.exists() and resolved.is_file():
            return resolved

    checked = "\n- ".join(str(path) for path in list(seen)[:10])
    raise FileNotFoundError(
        f"Could not find '{filename}'.\nChecked:\n- {checked}\n\n"
        "Tip: put the CSV in the same folder as this notebook, or run the notebook from your project root."
    )


museums_csv = find_local_csv("museums.csv")
state_pop_csv = find_local_csv("state_pop_data.csv")

print(f"Using museums file: {museums_csv}")
print(f"Using state population file: {state_pop_csv}")

df = pd.read_csv(museums_csv, low_memory=False)
df2 = pd.read_csv(state_pop_csv, low_memory=False)

In [ ]:
clean_copy = df.copy()
clean_copy = pd.merge(clean_copy,df2, on="State (Administrative Location)", how="outer")
clean_copy = clean_copy.drop(columns=["Street Address (Administrative Location)", "Latitude", "Longitude", "Street Address (Physical Location)","Zip Code (Physical Location)", "Zip Code (Administrative Location)", "Phone Number", "Alternate Name", "Locale Code (NCES)", "County Code (FIPS)", "State Code (FIPS)", "Region Code (AAM)","Employer ID Number"])
clean_copy["Tax Period"] = pd.to_datetime(clean_copy["Tax Period"], format="%Y%m")
clean_copy["Income"] = clean_copy["Income"].map('${:,.2f}'.format)
clean_copy["Revenue"] = clean_copy["Revenue"].map('${:,.2f}'.format)



In [ ]:
clean_copy["Tax Period"].min()

In [ ]:
clean_copy

In [ ]:
clean_copy.groupby('Museum Type').size()

In [ ]:
clean_copy.groupby('Museum Type')

In [ ]:
clean_copy.groupby('Museum Type').size().plot(kind='bar')

In [ ]:
zoo_aqua = clean_copy[clean_copy['Museum Type'].eq('ZOO, AQUARIUM, OR WILDLIFE CONSERVATION')].copy()

In [ ]:
zoo_aqua


In [ ]:
zoo = zoo_aqua['Museum Name'].str.contains('ZOO', case=False, na=False)
df_zoo = zoo_aqua[zoo]
df_zoo

In [ ]:
aqua = zoo_aqua['Museum Name'].str.contains('AQUARIUM', case=False, na=False)
df_aqua = zoo_aqua[aqua]
df_aqua

In [ ]:
pd.merge(df_zoo, df_aqua, how='outer')

In [ ]:
s_zoo_aqua = zoo_aqua.groupby('State (Administrative Location)').size().reset_index(name='Count').sort_values(by='Count', ascending=False)
s_zoo_aqua

In [ ]:
import matplotlib.pyplot as plt

# Show ALL states (no grouping into "Other").
pie_df = s_zoo_aqua[['State (Administrative Location)', 'Count']].copy()
pie_df = pie_df.sort_values('Count', ascending=False)

labels = pie_df['State (Administrative Location)']
values = pie_df['Count']

def pct_label(pct):
    # Keep chart readable: hide very tiny labels on slices.
    return f'{pct:.1f}%' if pct >= 2 else ''

fig, ax = plt.subplots(figsize=(11, 8))
wedges, _texts, autotexts = ax.pie(
    values,
    labels=None,
    autopct=pct_label,
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 0.8},
    pctdistance=0.75,
)

for text in autotexts:
    text.set_fontsize(9)

ax.set_title('Zoos/Aquariums by State (All States)')
ax.axis('equal')

# Wider legend with multiple columns so all states are visible.
ax.legend(
    wedges,
    labels,
    title='State',
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    ncol=2,
    fontsize=8,
    title_fontsize=10,
    frameon=True,
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

state_counts = (
    s_zoo_aqua[['State (Administrative Location)', 'Count']]
    .dropna()
    .sort_values('Count', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 14))
ax.barh(
    state_counts['State (Administrative Location)'],
    state_counts['Count'],
    color='#4C78A8'
)

ax.set_title('Number of Zoos and Aquariums by State', pad=12)
ax.set_xlabel('Count')
ax.set_ylabel('State')
ax.grid(axis='x', alpha=0.25)

for i, v in enumerate(state_counts['Count']):
    ax.text(v + 0.1, i, str(int(v)), va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

state_col = 'State (Administrative Location)'
type_col = 'Museum Type'

state_type_counts = (
    clean_copy.groupby([state_col, type_col])
    .size()
    .reset_index(name='Count')
)

states = sorted(state_type_counts[state_col].dropna().unique())

# Keep one fixed category order across all states.
type_order = (
    state_type_counts.groupby(type_col)['Count']
    .sum()
    .sort_values(ascending=True)
    .index
)

def plot_state_museum_types(selected_state):
    state_data = state_type_counts[state_type_counts[state_col] == selected_state]
    state_series = (
        state_data.set_index(type_col)['Count']
        .reindex(type_order, fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(state_series.index, state_series.values)
    ax.set_title(f'Museum Type Distribution in {selected_state}')
    ax.set_xlabel('Count')
    ax.set_ylabel('Museum Type')
    ax.grid(axis='x', alpha=0.2)
    plt.tight_layout()
    plt.show()

state_dropdown = widgets.Dropdown(
    options=states,
    value=states[0],
    description='State:',
    layout=widgets.Layout(width='350px')
)

interactive_plot = widgets.interactive_output(
    plot_state_museum_types,
    {'selected_state': state_dropdown}
)

display(state_dropdown, interactive_plot)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

heatmap_data = (
    clean_copy.groupby(['State (Administrative Location)', 'Museum Type'])
    .size()
    .unstack(fill_value=0)
)

# 1) Order columns by total frequency so most common museum types appear first.
ordered_cols = heatmap_data.sum(axis=0).sort_values(ascending=False).index
heatmap_data = heatmap_data[ordered_cols]

# 2) Use log scaling for color intensity while keeping raw counts as labels.
heatmap_log = np.log1p(heatmap_data)

# 3) Keep the chart readable: hide zeros and only annotate values >= 10.
annot_labels = heatmap_data.astype(str).mask(heatmap_data < 10, '')

plt.figure(figsize=(20, 14))
ax = sns.heatmap(
    heatmap_log,
    cmap='YlGnBu',
    linewidths=0.2,
    linecolor='white',
    annot=annot_labels,
    fmt='',
    cbar_kws={'label': 'log(1 + count)'}
)

plt.title('Museum Type Counts by State (Sorted, Log-Scaled, Labels >= 10)')
plt.xlabel('Museum Type (ordered by total count)')
plt.ylabel('State')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
revenue_num = pd.to_numeric(
    clean_copy['Revenue'].astype(str).str.replace(r'[\$,]', '', regex=True),
    errors='coerce'
)

top_revenue_museums = clean_copy.assign(Revenue_num=revenue_num)
top_revenue_museums = top_revenue_museums[top_revenue_museums['Revenue_num'].notna()]

top_value = top_revenue_museums['Revenue_num'].max()

top_revenue_museums = (
    top_revenue_museums[top_revenue_museums['Revenue_num'].eq(top_value)]
    [['Museum Name', 'State (Administrative Location)', 'Revenue_num']]
    .drop_duplicates()
    .sort_values(['Museum Name', 'State (Administrative Location)'])
)

top_revenue_museums['Revenue'] = top_revenue_museums['Revenue_num'].map('${:,.0f}'.format)
top_revenue_museums[['Museum Name', 'State (Administrative Location)', 'Revenue']]

In [ ]:

revs = clean_copy['Revenue']
revs = pd.to_numeric(
    revs.astype(str).str.replace(r'[\$,]', '', regex=True),
    errors='coerce'
)
revs.describe().map('${:,.2f}'.format)


In [ ]:
state_counts = clean_copy.groupby('State (Administrative Location)').size().reset_index(name='Count').sort_values(by='Count', ascending=False)
state_counts

In [ ]:
museum_types = clean_copy.groupby('Museum Type').size().reset_index(name='Count').sort_values(by='Count', ascending=False)
museum_types

In [ ]:
museum_type_revenue = (
    clean_copy.assign(Revenue_num=revenue_num)
    .groupby('Museum Type')['Revenue_num']
    .sum()
    .reset_index()
    .sort_values(by='Revenue_num', ascending=False)
)
museum_type_revenue['Revenue'] = museum_type_revenue['Revenue_num'].map('${:,.2f}'.format)
museum_type_revenue[['Museum Type', 'Revenue']]

In [ ]:
import matplotlib.ticker as mtick

ax = museum_type_revenue.plot(
    x='Museum Type',
    y='Revenue_num',
    kind='bar',
    legend=False,
    title='Total Revenue by Museum Type'
)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_ylabel('Revenue (USD)')

In [ ]:
stats_df = clean_copy.copy()

stats_df['Revenue_num'] = pd.to_numeric(
    stats_df['Revenue'].astype(str).str.replace(r'[\$,]', '', regex=True),
    errors='coerce'
)
stats_df['Income_num'] = pd.to_numeric(
    stats_df['Income'].astype(str).str.replace(r'[\$,]', '', regex=True),
    errors='coerce'
)

# Drop rows with NA or zero/non-positive values in either measure.
stats_df = stats_df.dropna(subset=['Revenue_num', 'Income_num'])
stats_df = stats_df[(stats_df['Revenue_num'] > 0) & (stats_df['Income_num'] > 0)]

# Drop outliers per museum type using the IQR rule for both revenue and income.
mask = pd.Series(True, index=stats_df.index)
for col in ['Revenue_num', 'Income_num']:
    q1 = stats_df.groupby('Museum Type')[col].transform(lambda s: s.quantile(0.25))
    q3 = stats_df.groupby('Museum Type')[col].transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask &= stats_df[col].between(lower, upper)

stats_no_outliers = stats_df[mask]

museum_type_describe = (
    stats_no_outliers.groupby('Museum Type')[['Revenue_num', 'Income_num']]
    .describe()
)

usd_stats = ['mean', 'std', 'min', '25%', '50%', '75%', 'max']
formatters = {
    ('Revenue_num', 'count'): '{:,.0f}'.format,
    ('Income_num', 'count'): '{:,.0f}'.format,
}
for metric in ['Revenue_num', 'Income_num']:
    for stat in usd_stats:
        formatters[(metric, stat)] = '${:,.2f}'.format

museum_type_describe.style.format(formatters)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import ipywidgets as widgets
from IPython.display import display

state_col = 'State (Administrative Location)'
museum_col = 'Museum Name'
rev_col = 'Revenue'

# Convert revenue to numeric and keep only valid positive values.
rev_numeric = pd.to_numeric(
    clean_copy[rev_col].astype(str).str.replace(r'[\$,]', '', regex=True),
    errors='coerce'
)

state_revenue = clean_copy[[state_col, museum_col]].copy()
state_revenue['Revenue_num'] = rev_numeric

# Remove missing/blank state and museum names, and remove NA/zero revenue.
state_revenue[museum_col] = state_revenue[museum_col].astype(str).str.strip()
state_revenue[state_col] = state_revenue[state_col].astype(str).str.strip()
state_revenue = state_revenue[
    state_revenue[museum_col].ne('')
    & state_revenue[state_col].ne('')
    & state_revenue['Revenue_num'].notna()
    & state_revenue['Revenue_num'].gt(0)
]

# If duplicates exist, combine them so a museum appears once per state.
state_revenue = (
    state_revenue.groupby([state_col, museum_col], as_index=False)['Revenue_num']
    .sum()
)

states = sorted(state_revenue[state_col].unique())
max_rev = float(state_revenue['Revenue_num'].max())

def plot_state_revenue(selected_state, min_revenue):
    all_state_data = (
        state_revenue[state_revenue[state_col] == selected_state]
        .sort_values('Revenue_num', ascending=True)
        .reset_index(drop=True)
    )

    if all_state_data.empty:
        fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
        ax.text(0.5, 0.5, 'No museums with revenue > 0 for this state.', ha='center', va='center')
        ax.axis('off')
        plt.show()
        return

    # Dynamic significance threshold: top quartile in selected state.
    state_q75 = all_state_data['Revenue_num'].quantile(0.75)
    threshold = max(float(min_revenue), float(state_q75))

    state_data = all_state_data[all_state_data['Revenue_num'] >= threshold]

    if state_data.empty:
        fig, ax = plt.subplots(figsize=(10, 3), constrained_layout=True)
        ax.text(
            0.5,
            0.5,
            f'No museums meet threshold (${threshold:,.0f}) in {selected_state}.',
            ha='center',
            va='center'
        )
        ax.axis('off')
        plt.show()
        return

    # Keep the figure compact and remove top/bottom padding in the plotting area.
    fig_height = max(4, len(state_data) * 0.28)
    fig, ax = plt.subplots(figsize=(12, fig_height), constrained_layout=True)
    ax.barh(state_data[museum_col], state_data['Revenue_num'], color='#2E86AB')
    ax.set_title(
        f'Significant Museum Revenue in {selected_state} (threshold >= ${threshold:,.0f})',
        pad=6
    )
    ax.set_xlabel('Revenue (USD)')
    ax.set_ylabel('Museum')
    ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
    ax.grid(axis='x', alpha=0.2)

    # Remove internal whitespace at top/bottom of the bar area.
    ax.margins(y=0)
    ax.set_ylim(-0.5, len(state_data) - 0.5)

    plt.show()

state_dropdown = widgets.Dropdown(
    options=states,
    value=states[0],
    description='State:',
    layout=widgets.Layout(width='320px')
)

min_revenue_slider = widgets.FloatLogSlider(
    value=1e6,
    base=10,
    min=3,
    max=max(3, int(len(str(int(max_rev))) - 1)),
    step=0.1,
    description='Min rev:',
    readout_format=',.0f',
    layout=widgets.Layout(width='420px')
)

interactive_plot = widgets.interactive_output(
    plot_state_revenue,
    {
        'selected_state': state_dropdown,
        'min_revenue': min_revenue_slider,
    }
)

controls = widgets.HBox([state_dropdown, min_revenue_slider])
display(controls, interactive_plot)

In [ ]:
#state_counts = clean_copy.groupby('State (Administrative Location)').size().reset_index(name='Count').sort_values(by='Count', ascending=False)
#state_counts["Count"] = state_counts["Count"].astype(int)
#state_pop = clean_copy.groupby('State (Administrative Location)').size().reset_index(name='Population (2008)').sort_values(by='State (Administrative Location)', ascending=True)
#state_pop["Population (2008)"] = state_pop["Population (2008)"].astype(int)
#per_capita = pd.merge(state_counts,state_pop, on="State (Administrative Location)", how="outer")
#state_pop
per_capita = clean_copy.groupby("State (Administrative Location)")["Population (2008)"].value_counts().reset_index(name="Count").sort_values(by="State (Administrative Location)", ascending=True)
per_capita["Count"] = per_capita["Count"].astype(int)
per_capita["Population (2008)"] = per_capita["Population (2008)"].astype(str).str.replace(r'[$\,]',"",regex=True)
per_capita["Population (2008)"] = per_capita["Population (2008)"].astype(int)
per_capita["per_capita"] = per_capita["Count"].div(per_capita["Population (2008)"])
per_capita.sort_values(by="per_capita", ascending=True)



In [ ]:
revenue_num = pd.to_numeric(clean_copy['Revenue'].astype(str).str.replace(r'[\$,]', '', regex=True),errors='coerce')
museum_type_revenue = (clean_copy.assign(Revenue_num=revenue_num).groupby('Museum Type')['Revenue_num'].sum().reset_index().sort_values(by='Revenue_num', ascending=False))
museum_type_revenue['Revenue'] = museum_type_revenue['Revenue_num'].map('${:,.2f}'.format)
museum_type_revenue[['Museum Type', 'Revenue']]
museum_type_revenue